<a href="https://colab.research.google.com/github/Nasim-Amani/ramp-aware-solar/blob/main/R_Q.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ramp-aware LSTM forecasting of solar irradiance (10–60 s ahead)

## Aim
We forecast solar irradiance 10 to 60 seconds ahead with an LSTM, and study how well the forecasts capture **ramps**: sudden changes in irradiance caused by passing clouds. Standard models are trained and judged with MSE/RMSE, which rewards smooth forecasts. Such models can beat persistence on RMSE and still miss most ramps, which are the events that matter most for grid and battery control.

## What the data exploration showed
- Ramps occur almost only on broken-cloud days, are real (confirmed by an independent PV module) and are fast: most are completed within 10 s, and the sun often stays blocked for only about 20 s.
- On clear days persistence is almost perfect up to one minute; on broken-cloud days its error doubles within the first minute.
- From the past, the **chance** of a ramp is predictable (restless periods last for minutes), but its **exact timing and direction** hardly are.

## Research questions
1. **Does a ramp-aware loss help?** Does adding a term on the size of the 10-s changes to the MSE loss improve ramp detection, and how much accuracy (RMSE skill) does it cost?
2. **Should the model predict the change instead of the value?** Does predicting the change from the current irradiance (and adding it to the current value) improve the forecasts, especially on clear days where persistence is almost perfect?
3. **Which inputs help?** Do time of day and measures of recent variability improve accuracy or ramp detection, compared with irradiance alone?
4. **What is the trade-off?** How do accuracy and ramp detection change as the weight λ of the ramp term increases?
5. **How robust are the results?** Do the conclusions hold for different lead times (10–60 s), for clear and broken-cloud days, and when each broken-cloud day is used as the test day?

## Setup
- **Data:** 1-s irradiance averaged to 10 s, daylight only, 11 complete days (16–26 Feb 2013).
- **Split by whole days, in time order:** train 16–23 Feb, validation 24 Feb, test 25 Feb (broken clouds) and 26 Feb (clear). Scaling uses the training days only.
- **Forecast:** input = the last 10 minutes; output = the next 6 values (10, 20, …, 60 s).
- **Model:** a single-layer LSTM (64 units), trained with early stopping. Each setup is trained with 3 random seeds and reported as mean ± standard deviation.

## Experiments

| | Question | What changes |
|---|---|---|
| **E1** | 1 and 2 | target (value vs change) × loss (MSE vs ramp-aware), irradiance only |
| **E2** | 3 | + time of day |
| **E3** | 3 | + recent-variability features |
| **E4** | 4 | weight λ of the ramp term (0, 0.5, 1, 3, 10) |
| **E5** | 5 | each broken-cloud day used as the test day in turn |

## Evaluation
- **Accuracy:** RMSE and skill against persistence, $\text{skill} = 1 - \text{RMSE}_{model} / \text{RMSE}_{persistence}$, for each lead time and separately for clear and broken-cloud days.
- **Ramps:** a ramp is a 10-s change larger than **80 W/m²** (95th percentile of the 10-s changes on the broken-cloud training days). We report precision, recall and F1 with a timing tolerance of ±10 s, on broken-cloud days. A threshold of 60 W/m² is used as a sensitivity check.